# Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, RocCurveDisplay, ConfusionMatrixDisplay
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Membaca data file CSV

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Pratikum_Ml/pratikum04/data/stunting_wasting_dataset.csv")
df.head()

# Melihat informasi umum dataset

In [ ]:
#melihat informasi umum data set
df.info ()

# Cek Missing Value

In [ ]:
#cek missing value
df.isnull().sum()

#Cek Nilai Unik

In [ ]:
#cek nilai uniq
df['Stunting'].unique()

In [ ]:
df['Jenis Kelamin'].unique()

# Mapping Kolom Kategorik ke Bentuk Numerik

In [ ]:
# 1. Mapping kolom Stunting -> biner
map_stunt = {'Stunted': 1, 'Severely Stunted': 1, 'Normal': 0, 'Tall': 0}
df['Stunting_bin'] = df['Stunting'].map(map_stunt).astype('Int64')

# 2. Mapping kolom Jenis Kelamin -> biner (Laki-laki = 1, Perempuan = 0)
df['JK_bin'] = (df['Jenis Kelamin'] == 'Laki-laki').astype(int)

print("Distribusi Stunting_bin: \n", df['Stunting_bin'].value_counts())
print("\nDistribusi JK_bin: \n", df['JK_bin'].value_counts())

# Analisis Korelasi Antar Variabel Numerik

In [ ]:
#Analisis Korelasi Antar Variabel Numerik
corr_matrix = df.corr(numeric_only=True)
corr_matrix

#  Visualisasi Heatmap Korelasi

In [ ]:
# Visualisasi heatmap
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(8,6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Heatmap Korelasi Antar Variabel Numerik terhadap Stunting", fontsize=12)
plt.show()

# Menentukan Fitur dan Target


In [ ]:
# Fitur numerik dan gender
feature_num = ['Umur (bulan)', 'Tinggi Badan (cm)', 'Berat Badan (kg)']
feature_bin = ['JK_bin']

# Gabungkan & drop missing (meskipun pada cek awal tidak ada missing value)
use_cols = feature_num + feature_bin + ['Stunting_bin']
df_model = df[use_cols].dropna().copy()

X = df_model[feature_num + feature_bin]
y = df_model['Stunting_bin']

print("X shape:", X.shape)
print("y shape:", y.shape)

# Pembagian Dataset (Training dan Testing)

In [ ]:
# Membagi Dataset menjadi Training dan Testing Set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("data latih:", X_train.shape)
print("data uji:", X_test.shape)

# Pembangunan Model Logistic Regression

In [ ]:
# Scale hanya fitur numerik, gender langsung passthrough
preprocess = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), feature_num),
        ('bin', 'passthrough', feature_bin)
    ],
    remainder='drop'
)

model = LogisticRegression(
    max_iter=1000,
    solver='lbfgs',
    class_weight='balanced',
    random_state=42
)

clf = Pipeline([
    ('preprocess', preprocess),
    ('model', model)
])

# Latih model
clf.fit(X_train, y_train)

print("Model Logistic Regression berhasil dilatih.")

 # Prediksi Model dan Evaluasi Model

In [ ]:
# Prediksi label dan probabilitas
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1] # Ambil probabilitas untuk kelas 1 (stunting)

# Hitung Metrik
print("Akurasi  : {:.4f}".format(accuracy_score(y_test, y_pred)))
print("Precision: {:.4f}".format(precision_score(y_test, y_pred, zero_division=0)))
print("Recall   : {:.4f}".format(recall_score(y_test, y_pred, zero_division=0)))
print("F1-Score : {:.4f}".format(f1_score(y_test, y_pred, zero_division=0)))
print("ROC-AUC  : {:.4f}".format(roc_auc_score(y_test, y_prob)))

#Visualisasi Hasil Evaluasi


In [ ]:
#visualisasi hasil evuluasi
# Tampilkan Confusion Matrix
ConfusionMatrixDisplay.from_estimator(clf, X_test, y_test,
                                      display_labels=['Tidak Stunting (0)', 'Stunting (1)'],
                                      cmap=plt.cm.viridis, normalize=None) # asumsi normalize=None
plt.title("Confusion Matrix")
plt.show()
#roc curve
RocCurveDisplay.from_estimator(clf, X_test, y_test)
plt.title("ROC Curve Logistic Regression")
plt.show()

# Classification Report


In [ ]:
#classification Report
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred,target_names=["Tidak Stunting (0)","Stunting (1)"]))

In [ ]:
from sklearn.model_selection import cross_val_score
#lakukan cross validation (cv=5 berati 5-fold)
scores =cross_val_score(clf,X,y, cv=5)
#tampilkan hasil
print("skor tial fold ",scores)
print("rata-rata skor ",np.mean(scores))
print("standar deviasi",np.std(scores))

# Interpretasi Model Logistic Regression

In [ ]:
# Ambil nama fitur dan koefisien dari pipeline
feat_names = feature_num + feature_bin
coefs = clf.named_steps['model'].coef_[0] # Ambil koefisien dari model di dalam pipeline
odds = np.exp(coefs)

# Buat DataFrame untuk interpretasi
coef_df = pd.DataFrame({
    'Fitur': feat_names,
    'Koefisien (log-odds)': coefs,
    'Odds Ratio (e^coef)': odds
}).sort_values('Odds Ratio (e^coef)', ascending=False)

display(coef_df)

#  Prediksi Data Baru (Contoh Kasus)


In [ ]:
#contoh 2 anak
data_baru = pd.DataFrame({
    "Umur (bulan)": [24, 10],
    "Tinggi Badan (cm)": [79.0, 72.5],
    "Berat Badan (kg)": [9.2, 7.8],
    "JK_bin": [1, 0] # 1=Laki-laki, 0=perempuan
})

pred = clf.predict(data_baru)
prob = clf.predict_proba(data_baru)[:,1]

hasil = data_baru.copy()
hasil['prob_Stunting']=prob
hasil['pred (0=Tidak,1=ya)'] =pred
display(hasil)

# Tugas Praktikum Mandiri

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, RocCurveDisplay


In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Pratikum_Ml/pratikum04/data/calonpembelimobil.csv", sep=",")
df.head()

In [ ]:
# Mengecek struktur dan nilai kosong
df.info()
print("\nCek data kosong tiap kolom:\n", df.isnull().sum())


In [ ]:
X = df[['Usia', 'Status', 'Kelamin', 'Memiliki_Mobil', 'Penghasilan']]
y = df['Beli_Mobil']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Akurasi Model:", accuracy_score(y_test, y_pred))
print("\nLaporan Klasifikasi:\n", classification_report(y_test, y_pred))

In [ ]:
data_baru = pd.DataFrame({
    'Usia': [30],
    'Status': [1],
    'Kelamin': [0],
    'Memiliki_Mobil': [0],
    'Penghasilan': [250]
})

prediksi = model.predict(data_baru)
print("Prediksi Beli Mobil:", "Ya" if prediksi[0] == 1 else "Tidak")